# Lab 6: Markov Chains and Customer Loyalty

Complete the code TODOs, the required written responses, the manager recommendation, the reflection, and the AI-use statement. Think about the discussion questions and be prepared to explain your reasoning to the TA. Every code location that you must complete is marked `TODO`; code marked **Provided** or **do not edit** should be left unchanged. 

## Student Information

**Name:** Write your answer here

**Student ID:** Write your answer here

**Date:** Write your answer here

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Provided lab settings -- do not edit.
SEED = 211                                          # seed for every random result in this lab
states = np.array(["Loyal", "Occasional", "Inactive"])
WEEKS = 12                                          # weeks simulated in Parts B and E
N_CUSTOMERS = 1000                                  # customers simulated in Part B

## Part A: The Transition Matrix

Chapter 8 models a customer's loyalty as a Markov chain: the probabilities for next week's state depend only on this week's state. Rows are current states, columns are next week's states, and each row is a probability distribution. With the row-vector convention,

$$
\pi_{n+1}=\pi_nP,\qquad \pi_n=\pi_0P^n .
$$

| From / To | Loyal | Occasional | Inactive |
|---|---:|---:|---:|
| Loyal | 0.70 | 0.20 | 0.10 |
| Occasional | 0.25 | 0.55 | 0.20 |
| Inactive | 0.10 | 0.25 | 0.65 |

In [ ]:
# Enter the baseline transition matrix (rows in the order of `states`).
P = TODO

In [ ]:
def validate_transition_matrix(matrix):
    # Return True only if the matrix is square, every entry is in [0, 1], and every row sums to 1.
    matrix = np.asarray(matrix, dtype=float)
    is_square = TODO
    in_bounds = TODO
    rows_sum_to_one = TODO
    return is_square and in_bounds and rows_sum_to_one


# Provided trial checks -- do not edit.
bad_sum = P.copy()
bad_sum[0, 0] += 0.20                       # Loyal row now sums to 1.2
bad_negative = P.copy()
bad_negative[1] = [-0.10, 0.70, 0.40]       # sums to 1 but contains a negative entry
assert validate_transition_matrix(P)
assert not validate_transition_matrix(bad_sum)
assert not validate_transition_matrix(bad_negative)
print("Valid matrix accepted; both invalid trials rejected.")

Think about the following question:

1. What does the Loyal row mean?

2. The `bad_negative` row sums to 1. Why is it still not a valid row?

3. State the Markov assumption for this model in one sentence. Give one real reason it might fail.

4. What information about a real customer is lost when we use only three states?

## Part B: Simulate Customers

### One step from one uniform number

Chapter 8 simulates one step by drawing $U\sim\mathrm{Uniform}(0,1)$ and comparing it with the cumulative probabilities of the current row. This is the **discrete inverse-transform method** from Lab 3 Part D: the next state is the first cut point at or above $U$. For the Loyal row the cut points are $0.70,\ 0.90,\ 1.00$, so $U=0.81$ gives Occasional.

Complete `sample_state`, then `simulate_path`. Do **not** use `rng.choice`.

In [ ]:
def sample_state(probabilities, u):
    # Return the index of the first cumulative probability that is >= u.
    cut_points = np.cumsum(probabilities)
    cut_points[-1] = 1.0                    # Provided: protects against rounding, e.g. 0.9999999
    return TODO


# Provided hand checks -- do not edit.
assert sample_state(P[0], 0.69) == 0       # Loyal row: 0.69 <= 0.70
assert sample_state(P[0], 0.81) == 1       # 0.70 < 0.81 <= 0.90
assert sample_state(P[0], 0.95) == 2       # 0.90 < 0.95
print("sample_state passes the hand checks.")

In [ ]:
def simulate_path(matrix, start, n_steps, rng):
    # Return an array of n_steps + 1 state indices: the start state, then one state per step.
    path = [start]
    current = start
    for _ in range(n_steps):
        u = rng.random()
        current = TODO
        path.append(current)
    return np.array(path)


one_path = TODO    # start Loyal, 12 weeks, fresh generator np.random.default_rng(SEED)

# Provided display -- do not edit.
assert len(one_path) == WEEKS + 1
pd.Series(states[one_path], name="state").rename_axis("week").to_frame().T

### Many customers, and a check against the matrix

The shop's current customer mix is $\pi_0=(0.50,\ 0.35,\ 0.15)$. Each simulated customer draws a starting state from $\pi_0$ (again with `sample_state`) and then follows the chain for 12 weeks.

In [ ]:
pi0 = np.array([0.50, 0.35, 0.15])


def simulate_customers(matrix, initial, n_customers, n_steps, rng):
    # Return an (n_customers, n_steps + 1) array of state indices.
    paths = []
    for _ in range(n_customers):
        start = TODO
        paths.append(TODO)
    return np.array(paths)


customer_paths = TODO    # 1,000 customers from pi0, 12 weeks, fresh generator np.random.default_rng(SEED)

# Exact distribution pi0 P^n for n = 0, 1, ..., WEEKS (one row per week).
forecast = TODO

# Provided comparison -- do not edit.
simulated = np.array([[np.mean(customer_paths[:, week] == s) for s in range(3)] for week in range(WEEKS + 1)])
assert customer_paths.shape == (N_CUSTOMERS, WEEKS + 1)
pd.DataFrame([simulated[WEEKS], forecast[WEEKS]],
             index=["Simulated, week 12", "Exact pi0 P^12"], columns=states).round(4)

In [ ]:
# Provided plot -- do not edit.
fig, ax = plt.subplots(figsize=(9, 4))
for s, name in enumerate(states):
    ax.plot(range(WEEKS + 1), simulated[:, s], marker="o", label=f"{name} (simulated)")
    ax.plot(range(WEEKS + 1), forecast[:, s], color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Week")
ax.set_ylabel("Proportion of customers")
ax.set_title("1,000 simulated customers (markers) and exact pi0 P^n (dashed)")
ax.legend()
plt.show()

### Part B Questions

Think about the following questions:

1. The single path left Loyal and later came back. Is that typical? What does one path tell a manager, and what does it not?
2. Why do the simulated week-12 proportions differ slightly from $\pi_0P^{12}$? Which of the two is random?
3. Why does `simulate_path` with 12 steps return 13 states?

## Checkpoint 1: Matrix and Simulation (1 of 3 Points)

Show your notebook to the TA after completing Parts A-B. To receive the first 1 point, your notebook must contain:

1. the validated matrix, the passed trial checks, and the three Part A responses;
2. `sample_state` passing the hand checks and the reproducible 12-week path; and
3. the 1,000-customer comparison table and plot.

Be prepared to explain how one uniform number becomes the next state, and why the simulated and exact week-12 proportions differ. Continue with Parts C-E after the checkpoint.

## Part C: Long-Run Behaviour

Chapter 8: a **stationary distribution** satisfies $\pi=\pi P$ with $\sum_j\pi_j=1$. If the chain is irreducible and aperiodic (as here), then

1. the rows of $P^n$ all converge to $\pi$, whatever the starting state; and
2. the fraction of time **one long path** spends in state $j$ converges to $\pi_j$ (the ergodic theorem).

We estimate $\pi$ three ways: matrix powers, solving $\pi=\pi P$, and one long simulated path. The table also shows the week-12 proportions of the 1,000 customers from Part B: many independent customers observed at one late time.

In [ ]:
# Matrix powers P^1, P^2, P^5, P^10, P^20 in a dictionary {n: matrix}.
powers = TODO

# Provided display -- do not edit.
for n, matrix in powers.items():
    print(f"P^{n} =\n{np.round(matrix, 4)}\n")

**Solving $\pi=\pi P$.** Transposing gives $(P^{\mathsf T}-I)\pi^{\mathsf T}=0$. These equations are not independent, so we replace the last one by $\pi_1+\pi_2+\pi_3=1$ and solve the linear system.

In NumPy, `matrix.T` is the transpose $P^{\mathsf T}$ and `np.eye(n)` is the $n\times n$ identity matrix $I$. The transpose is needed because `np.linalg.solve(A, b)` solves $Ax=b$ for a **column** vector $x$, while $\pi=\pi P$ is written for a **row** vector.

In [ ]:
def stationary_distribution(matrix):
    n = len(matrix)
    A = TODO          # coefficient matrix of (P^T - I) pi^T = 0
    A[-1] = 1                           # Provided: replace the last equation by sum(pi) = 1
    rhs = np.zeros(n)
    rhs[-1] = 1
    return np.linalg.solve(A, rhs)


pi = stationary_distribution(P)
assert np.allclose(pi @ P, pi), "pi should satisfy pi P = pi"

# One long path: 20,000 weeks from Loyal. Discard weeks 0-100 as warm-up, then
# compute the fraction of the remaining weeks spent in each state.
long_path = TODO    # fresh generator np.random.default_rng(SEED)
kept = long_path[101:]
time_fractions = TODO

# Provided comparison -- do not edit.
pd.DataFrame([powers[20][0], pi, time_fractions, simulated[WEEKS]],
             index=["Row 1 of P^20", "Solve pi = pi P", "One long path (time fractions)",
                    "1,000 customers in week 12 (Part B)"],
             columns=states).round(4)

### Part C Questions

1. What happens to the rows of $P^n$ as $n$ grows, and what does that mean for a customer who is Loyal today?

   **Response 1:** Write your answer here.

2. Compare the four rows. Which are exact once $P$ is fixed, and which are random? Why can one long path, or many customers observed in week 12, estimate $\pi$ at all?

   **Response 2:** Write your answer here.

Think about the following question:

3. If Inactive $\to$ Inactive were 0.95 instead of 0.65, which way would the long-run Inactive proportion move?

## Part D: Warm-Up and Dependence (Chapter 8, slotted coffee-shop queue)

Chapter 8 §8.11 turns the coffee-shop queue into a Markov chain by looking once per minute. $X_n\in\{0,1,2,3\}$ is the number of customers in the shop at the end of minute $n$; an arrival happens with probability $p=\tfrac14$ and a busy barista finishes with probability $q=\tfrac12$. The matrix and its exact stationary distribution are

$$
Q=\begin{pmatrix}
\tfrac34&\tfrac14&0&0\\[2pt]
\tfrac38&\tfrac12&\tfrac18&0\\[2pt]
0&\tfrac38&\tfrac12&\tfrac18\\[2pt]
0&0&\tfrac38&\tfrac58
\end{pmatrix},\qquad
\pi=\frac{1}{53}(27,18,6,2),\qquad \sum_n n\pi_n=\frac{36}{53}\approx0.679 .
$$

Many simulated runs should reproduce the solved $\pi$ approximately. The notes also give two warnings: runs that start empty are biased at first, and the minutes inside one run are **dependent**, so $s/\sqrt N$ from one run understates the uncertainty. Check the first claim, then test both warnings.

In [ ]:
# Provided matrix -- do not edit.
Q = np.array([
    [3/4, 1/4, 0,   0  ],
    [3/8, 1/2, 1/8, 0  ],
    [0,   3/8, 1/2, 1/8],
    [0,   0,   3/8, 5/8],
])
assert validate_transition_matrix(Q)
assert np.allclose(stationary_distribution(Q), np.array([27, 18, 6, 2]) / 53)   # your functions reproduce the notes
exact_mean = 36 / 53

In [ ]:
# 200 independent runs of 1,000 minutes, each starting from an empty shop (state 0).
queue_rng = np.random.default_rng(SEED)        # ONE generator: each run continues the stream
n_runs, n_minutes = 200, 1000
run_averages, naive_ses, all_paths = [], [], []

for run in range(n_runs):
    path = TODO
    x = path[1:]                                       # X_1, ..., X_1000
    run_averages.append(TODO)
    naive_ses.append(TODO)   # the Chapter 6 formula, applied to ONE run
    all_paths.append(path)                             # Provided: keep the whole run X_0, ..., X_1000

run_averages = np.array(run_averages)
all_paths = np.array(all_paths)                        # Provided: one row per run, shape (200, 1001)

# Provided summary -- do not edit.
pd.Series({
    "exact long-run mean 36/53": exact_mean,
    "mean of 200 run averages": run_averages.mean(),
    "typical naive SE s/sqrt(N) from one run": np.mean(naive_ses),
    "actual SD of the 200 run averages": run_averages.std(ddof=1),
}).round(4)

In [ ]:
# Provided: many runs versus the solved stationary distribution -- do not edit.
pd.DataFrame([
    [np.mean(all_paths[:, 1:] == k) for k in range(4)],    # every minute of every run
    [np.mean(all_paths[:, -1] == k) for k in range(4)],    # minute 1000 only: one value per run
    stationary_distribution(Q),
], index=["All minutes of 200 runs", "Minute 1000 of 200 runs", "Solved pi = pi Q"],
   columns=[0, 1, 2, 3]).rename_axis(columns="customers in shop").round(4)

In [ ]:
# Provided warm-up plot -- do not edit.
minute_means = all_paths[:, :51].mean(axis=0)            # mean of X_0, ..., X_50 over the 200 runs
plt.figure(figsize=(9, 4))
plt.plot(range(51), minute_means, marker=".", label="mean of X_n over 200 runs (start empty)")
plt.axhline(exact_mean, color="black", linestyle="--", label="long-run mean 36/53")
plt.xlabel("minute n")
plt.ylabel("customers in shop")
plt.title("The first minutes remember the empty start")
plt.legend()
plt.show()

### Part D Questions

1. Compare the naive standard error from one run with the actual standard deviation of the 200 run averages. Which should you trust, and why are they different?

   **Response 1:** Write your answer here.

2. What does the warm-up plot show? How does it explain why a short run that starts empty underestimates the long-run mean (compare Lab 5's rush scenario)?

   **Response 2:** Write your answer here.

Think about the following questions:

3. The mean of the 200 run averages is close to $36/53$ even though every run starts empty. Why is the warm-up bias small here but large in Lab 5's rush scenario?

4. In the table of state proportions, which result justifies each simulated row? The first row is closer to $\pi$ than the second, even though its minutes are dependent. Why?

## Part E: Manager Challenge — a Loyalty Card

The manager proposes a loyalty card and asks what it would do to the customer mix. The marketing team's **assumed** effect is this matrix:

| From / To | Loyal | Occasional | Inactive |
|---|---:|---:|---:|
| Loyal | 0.80 | 0.15 | 0.05 |
| Occasional | 0.35 | 0.50 | 0.15 |
| Inactive | 0.15 | 0.30 | 0.55 |

These are exact model quantities, so no new simulation is needed: **simulate when necessary; calculate analytically when the model gives an exact quantity** (Lab 3).

In [ ]:
P_card = TODO
assert validate_transition_matrix(P_card)


def distribution_after(matrix, initial, n):
    # Exact distribution after n weeks: initial P^n.
    return TODO


# Provided comparison -- do not edit.
rows = []
for name, matrix in [("Baseline", P), ("Loyalty card", P_card)]:
    for week in [1, 4, 12]:
        rows.append({"case": name, "horizon": f"week {week}", **dict(zip(states, distribution_after(matrix, pi0, week)))})
    rows.append({"case": name, "horizon": "long run", **dict(zip(states, stationary_distribution(matrix)))})
pd.DataFrame(rows).set_index(["case", "horizon"]).round(4)

In [ ]:
# Provided: change ONE row of the baseline to its loyalty-card row and see the long-run Loyal share -- do not edit.
one_row_rows = []
for r, name in enumerate(states):
    changed = P.copy()
    changed[r] = P_card[r]
    one_row_rows.append({"row changed": name, "long-run Loyal": stationary_distribution(changed)[0]})
one_row_rows.append({"row changed": "none (baseline)", "long-run Loyal": pi[0]})
one_row_rows.append({"row changed": "all three (card)", "long-run Loyal": stationary_distribution(P_card)[0]})
pd.DataFrame(one_row_rows).set_index("row changed").round(4)

### Part E Questions

1. Describe the effect of the card on the Loyal share over time. How quickly does most of the effect appear?

   **Response 1:** Write your answer here.

2. Which single row change matters most for the long-run Loyal share? Why might that be?

   **Response 2:** Write your answer here.

Think about the following question:

3. The card matrix is an assumption. What data would you need before trusting it?

## Recommendation, Reflection, and AI Use

**Manager recommendation:** Use the Part E evidence. State what the model predicts, how quickly, and what evidence is needed before rolling the card out. Write your answer here.

**Reflection:** What does the Markov assumption mean here? Why is one customer path not enough? What did matrix powers and the stationary distribution add? What did Part D show about simulating long runs? Write your answer here.

**AI-use statement:** If AI was used, identify the tool, how it was used, and how you checked or corrected its output. If AI was not used, say so. You must be able to explain all submitted work. Write your answer here.

## Checkpoint 2: Final Submission

Show the completed Parts C-E, recommendation, reflection, and AI-use statement to the TA. Be prepared to answer one code question and one modelling question, for example why the naive standard error in Part D is too small. After the final checkpoint:

1. save the notebook as `Lab06_StudentID.ipynb`, for example `Lab06_6900789.ipynb`;
2. restart the kernel and run all cells from top to bottom;
3. check that there are no errors;
4. save the notebook; and
5. upload the single completed `.ipynb` file to the **Lab 06** assignment in Google Classroom before leaving the lab.

## Optional Bonus (not assessed): Markov Chain Monte Carlo

MCMC builds a Markov chain whose **stationary distribution is a target we want to sample from**. Target drink preferences: Coffee 0.50, Latte 0.30, Matcha 0.20. From the current drink, propose one of the other two at random and accept the move with probability $\min\bigl(1,\ \text{target[new]}/\text{target[current]}\bigr)$. Run 21,000 steps from Coffee, discard the first 1,000, and compare the sampled proportions with the target.

In [ ]:
target = np.array([0.50, 0.30, 0.20])
mcmc_rng = np.random.default_rng(SEED)
current = 0
samples = []
for _ in range(21_000):
    proposal = [k for k in range(3) if k != current][int(mcmc_rng.random() < 0.5)]   # one of the other two, equally likely
    if mcmc_rng.random() < TODO:
        current = proposal
    samples.append(current)

mcmc_proportions = np.bincount(samples[1000:], minlength=3) / 20_000
pd.DataFrame({"target": target, "MCMC": mcmc_proportions}, index=["Coffee", "Latte", "Matcha"]).round(4)